# Create a RAG applications using typesense

In [1]:
import typesense

In [ ]:
client = typesense.Client({
    'nodes':[{
        'host':'', # ENTER TYPESENSE HOST        
        'port':'', # ENTER TYPESENSE PORT
        'protocol':'' # ENTER TYPESENSE PROTOCOL
    }],
    'api_key':'',   # ENTER TYPESENSE API KEY
    'connection_timeout_seconds':2
})


books_schema = {
    'name':'books',
    'fields':[
        {'name':'title' , 'type' :'string'},
        {'name':'authors' , 'type' : 'string[]' , 'facet':True},
        {'name':'publication_year' , 'type' : 'int32' , 'facet':True},
        {'name':'ratings_count' , 'type' : 'int32'},
        {'name':'average_rating' , 'type' : 'float'}
    ],
    'default_sorting_field':'ratings_count'
}

print(client.collections.create(books_schema))

In [ ]:
with open('books.jsonl', 'r' , encoding='utf-8')as jsonl_file:
    data = jsonl_file.read()
    client.collections['books'].documents.import_(data)

In [ ]:
search_parameters = {
    'q' : 'harry potter',
    'query_by' : "title , authors",
    'sort_by': 'ratings_count:desc'
}

client.collections['books'].documents.search(search_parameters)

In [ ]:
search_parameters = {
    'q' : 'harry potter',
    'query_by' : "title",
    'filter_by' : 'publication_year : <1998',
    'sort_by' : 'publication_year:desc'
}

client.collections['books'].documents.search(search_parameters)

In [ ]:
search_parameters = {
    'q' : 'experyment',
    'query_by' : "title",
    'facet_by' : 'authors',
    'sort_by' : 'average_rating:desc'
}

client.collections['books'].documents.search(search_parameters)

## Langchain + typesense + groq_llm + RAG Applications

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

c:\Desktop\RAG-Krish Nayak\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import os
os.environ["GROQ_API_KEY"]="" #ENTER GROQ_API_KEY
os.environ["HF_TOKEN"]= "" #ENTER HUGGING_FACE_TOKEN

In [6]:
loader = TextLoader("test.text")
documents = loader.load()

text_splitter = CharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 0)
docs = text_splitter.split_documents(documents)

embeddings = HuggingFaceEmbeddings()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5166.52it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
docsearch = Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        'host':'', # ENTER TYPESENSE HOST
        'port':'',  # ENTER TYPESENSE PORT
        'protocol':'https',  # ENTER TYPESENSE PROTOCOL
        'typesense_api_key':"", # ENTER TYPESENSE API KEY
        'typesense_collection_name':"lang-chain"
    }
)

In [18]:
query = "what is artificial intelligence"
found_doc = docsearch.similarity_search(query)
# print(found_doc)
print(found_doc[0].page_content)

What Is Artificial Intelligence? Definition, Examples & Benefits Explained
Artificial Intelligence (AI) is a groundbreaking technology shaping the future of industries and everyday life. From advanced robotics and self-driving cars to virtual assistants and personalised recommendations, AI is revolutionising the way we work, live, and interact. This page consists of essays on artificial intelligence tailored to different lengths and needs, including 100, 150, 250, and 300-word formats. These essays provide a clear understanding of AIâ€™s transformative potential, challenges, and significance in the modern world. Whether you're looking for a short overview or a detailed analysis, this page covers it all.

Artificial Intelligence Essay in 100 words
Artificial Intelligence is about making machines that can think and learn like people. These machines are made to solve problems, make decisions, and do tasks like recognising voices, reading data, or driving cars.


In [19]:
# Retriever

retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x0000026DE9A60EB0>, search_kwargs={})

In [21]:
query = "what is artificial intelligence"
print(retriever.invoke(query)[0])

page_content='What Is Artificial Intelligence? Definition, Examples & Benefits Explained
Artificial Intelligence (AI) is a groundbreaking technology shaping the future of industries and everyday life. From advanced robotics and self-driving cars to virtual assistants and personalised recommendations, AI is revolutionising the way we work, live, and interact. This page consists of essays on artificial intelligence tailored to different lengths and needs, including 100, 150, 250, and 300-word formats. These essays provide a clear understanding of AIâ€™s transformative potential, challenges, and significance in the modern world. Whether you're looking for a short overview or a detailed analysis, this page covers it all.

Artificial Intelligence Essay in 100 words
Artificial Intelligence is about making machines that can think and learn like people. These machines are made to solve problems, make decisions, and do tasks like recognising voices, reading data, or driving cars.' metadata={'so